In [18]:
tabla='qtsoo10'
col_tabla = 'solopefec'

In [19]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta
import time 
from datetime import datetime
from sqlalchemy import text
import oracledb
import sys
import psycopg2

In [20]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

In [21]:
# fecha = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=13", con=connection2)
# fecha= fecha.iloc[0, 0]
# print(fecha)

now = datetime.now()

# query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=13"

# c1= text(query)
# connection2.execute(c1)

In [22]:
fecha = '01/05/2023'

In [23]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection0 = engine0.connect()

# query0 = f"SELECT * FROM (SELECT * FROM {tabla} WHERE ROWNUM <= 10)"

query0 = f"""
select
d1.SOLOPEORICENASICOD,  
d1.SOLOPECENASICOD,  
d1.SOLOPENUM,
d1.SOLOPECPSCOD,
d1.SOLOPEFLGPRI,
d1.SOLOPEPQXCPSCOD,
d1.SOLOPEPQXSECUEN,
d1.SOLOPESOLMATDOCNUM,
d1.SOLOPEGRDCOMOPECOD,a1.solopefec as solopefec,  
a1.SOLOPEACTMEDNUM,  
a1.SOLOPEBUSPACSECNUM
from {tabla} d1  
left outer join qtsop10 a1 on a1.SOLOPEORICENASICOD = d1.SOLOPEORICENASICOD  
and a1.SOLOPECENASICOD = d1.SOLOPECENASICOD  
and a1.SOLOPENUM = d1.SOLOPENUM
where a1.{col_tabla}>='{fecha}'

"""

base1 = pd.read_sql_query(query0,con=connection0)


connection0.close()

In [24]:
base1.shape

(100211, 12)

In [25]:
base1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100211 entries, 0 to 100210
Data columns (total 12 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   solopeoricenasicod  100211 non-null  object        
 1   solopecenasicod     100211 non-null  object        
 2   solopenum           100211 non-null  int64         
 3   solopecpscod        100211 non-null  object        
 4   solopeflgpri        65456 non-null   float64       
 5   solopepqxcpscod     65456 non-null   object        
 6   solopepqxsecuen     65456 non-null   float64       
 7   solopesolmatdocnum  65456 non-null   float64       
 8   solopegrdcomopecod  21364 non-null   object        
 9   solopefec           100211 non-null  datetime64[ns]
 10  solopeactmednum     100211 non-null  int64         
 11  solopebuspacsecnum  100211 non-null  int64         
dtypes: datetime64[ns](1), float64(3), int64(3), object(5)
memory usage: 9.2+ MB


In [26]:
base1.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

211

In [27]:
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL
query_count_before = text(f"SELECT COUNT(*) FROM {tabla}")
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla qtsoo10 antes de la inserción: 1599268


In [28]:
# resultado = base1.query('solopecpscod == solopepqxcpscod and solopecpscod.notnull() and solopepqxcpscod.notnull()')
# resultado

In [29]:
#Borramos en caso el ETL se ejecute una segunda vez
borrando=text(f"DELETE FROM {tabla} WHERE {col_tabla} >='{fecha}'")
borrado = connection3.execute(borrando)

In [30]:
query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN solopeoricenasicod TYPE character(1),
ALTER COLUMN solopecenasicod TYPE character(3),
ALTER COLUMN solopenum TYPE numeric(10,0),
ALTER COLUMN solopecpscod TYPE character(10),
ALTER COLUMN solopeflgpri TYPE numeric(1,0),
ALTER COLUMN solopepqxcpscod TYPE character(10),
ALTER COLUMN solopepqxsecuen TYPE numeric(2,0),
ALTER COLUMN solopesolmatdocnum TYPE numeric(10,0),
ALTER COLUMN solopegrdcomopecod TYPE character(1),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);


INSERT INTO {tabla} 
(solopeoricenasicod,solopecenasicod,solopenum,solopecpscod,solopeflgpri,solopepqxcpscod,solopepqxsecuen,solopesolmatdocnum,
solopegrdcomopecod,solopefec,solopeactmednum,solopebuspacsecnum) 

SELECT 
solopeoricenasicod,solopecenasicod,solopenum,solopecpscod,solopeflgpri,solopepqxcpscod,solopepqxsecuen,solopesolmatdocnum,
solopegrdcomopecod,solopefec,solopeactmednum,solopebuspacsecnum


FROM tmp_{tabla};
"""

c1= text(query)
connection3.execute(c1)

In [31]:
query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla qtsoo10 después de la inserción: 1599291
La cantidad de filas insertadas fue de: 23


In [32]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""
c2= text(query2)
cursor=connection3.execute(c2)

In [33]:
connection1.close()
connection2.close()
connection3.close()

AYUDA PARA EXTRAER COLUMNAS Y ESTRUCTURA DE TABLAS (NO ES PARTE DEL ETL)

In [34]:
import re

cadena = """
    solopeoricenasicod character(1) COLLATE pg_catalog."default",
    solopecenasicod character(3) COLLATE pg_catalog."default",
    solopenum numeric(10,0),
    solopecpscod character(10) COLLATE pg_catalog."default",
    solopeflgpri numeric(1,0),
    solopepqxcpscod character(10) COLLATE pg_catalog."default",
    solopepqxsecuen numeric(2,0),
    solopesolmatdocnum numeric(10,0),
    solopegrdcomopecod character(1) COLLATE pg_catalog."default",
    solopefec date,
    solopeactmednum numeric(10,0),
    solopebuspacsecnum numeric(10,0)
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ",".join(nombrecitos)

print(insertado_col)

ALTER COLUMN solopeoricenasicod TYPE character(1),
ALTER COLUMN solopecenasicod TYPE character(3),
ALTER COLUMN solopenum TYPE numeric(10,0),
ALTER COLUMN solopecpscod TYPE character(10),
ALTER COLUMN solopeflgpri TYPE numeric(1,0),
ALTER COLUMN solopepqxcpscod TYPE character(10),
ALTER COLUMN solopepqxsecuen TYPE numeric(2,0),
ALTER COLUMN solopesolmatdocnum TYPE numeric(10,0),
ALTER COLUMN solopegrdcomopecod TYPE character(1),
ALTER COLUMN solopefec TYPE date,
ALTER COLUMN solopeactmednum TYPE numeric(10,0),
ALTER COLUMN solopebuspacsecnum TYPE numeric(10,0);

solopeoricenasicod,solopecenasicod,solopenum,solopecpscod,solopeflgpri,solopepqxcpscod,solopepqxsecuen,solopesolmatdocnum,solopegrdcomopecod,solopefec,solopeactmednum,solopebuspacsecnum
